
# FAALProt Heterogeneity Analysis — Jupyter Edition

Complete, publication‑grade pipeline to quantify and visualize heterogeneity in FAAL proteins.

**New option:** choose to (a) place UMAP info *inside the image* (top‑right), (b) include it *only in the filename*, (c) both, or (d) neither.


In [1]:
# =============================
# Parameters (edit as needed)
# =============================
FASTA_PATH = "FAAL_NR_MIBIG_LEGE_FFT_NS_2_raw_data.fasta"  # FASTA in current directory
TABLE_S2_PATH = "Table_S2.tsv"                              # Optional; if absent, phylum = Unknown
OUTPUT_DIR = "results_faal_jupyter"

# Sequence subset control (use either SUBSET_SIZE or SUBSET_FRACTION or leave both as None for full)
SUBSET_SIZE = None           # e.g., 5000
SUBSET_FRACTION = None       # e.g., 0.20
SUBSET_SEED = 42

# Pairwise identity sampling (number of pairs to keep by deterministic stride)
PAIR_SAMPLE_SIZES = [600_000, 1_000_000]

# Optional Part B: Word2Vec on k-mers from aligned sequences without gaps
RUN_PART_B = True
W2V_K = 3
W2V_VECTOR_SIZE = 390
W2V_WINDOW = 5
W2V_MIN_COUNT = 1
W2V_SG = 1
W2V_EPOCHS = 2500
W2V_WORKERS = 4

# Binary locations (auto-detected by default; override if needed)
MAFFT_BIN = None   # e.g., "/usr/bin/mafft"
MMSEQS_BIN = None  # e.g., "/home/USER/anaconda3/envs/fall/bin/mmseqs"

# --- NEW: Annotation/filename options for UMAPs ---
# Choose how to include distance method + sampling info:
#   ANNOTATE_ON_CANVAS: if True, text box appears in top-right of the figure
#   APPEND_INFO_TO_FILENAME: if True, info is appended to output filename
ANNOTATE_ON_CANVAS = True
APPEND_INFO_TO_FILENAME = True


In [2]:

# =============================
# Imports and global constants
# =============================
import os, sys, json, shutil, random, warnings, subprocess, re
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, List, Tuple, Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Optional dependencies
try:
  from Bio import SeqIO
  HAS_BIO = True
except Exception:
  HAS_BIO = False

# UMAP (optional)
try:
  import umap
  HAS_UMAP = True
except Exception:
  try:
    from umap import UMAP as _UMAP
    class umap:
      UMAP = _UMAP
    HAS_UMAP = True
  except Exception:
    HAS_UMAP = False

from sklearn.manifold import TSNE
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform
try:
  from gensim.models import Word2Vec
  HAS_GENSIM = True
except Exception:
  HAS_GENSIM = False

RANDOM_STATE = 42
FIXED_CUTOFFS = [10,20,30,40,50,60,70,80,90]
DOMAIN_TOKENS = {'bacteria','archaea','eukaryota','eukarya','viruses','virus','viroids'}


/home/mattoslmp/anaconda3/envs/fall/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

# =============================
# Utilities & I/O
# =============================
def ensure_output_dir(path: str) -> None:
  os.makedirs(path, exist_ok=True)

def _normalize_actino(s: Optional[str]) -> str:
  if not isinstance(s, str):
    return "Unknown"
  s = s.strip()
  return "Actinomycetota" if s == "Actinobacteria" else (s if s else "Unknown")

def read_fasta_minimal(fasta_path: str) -> pd.DataFrame:
  identifiers, sequences = [], []
  with open(fasta_path, 'r', encoding='utf-8') as fh:
    current_id, buff = None, []
    for line in fh:
      line = line.strip()
      if not line:
        continue
      if line.startswith('>'):
        if current_id is not None:
          sequences.append(''.join(buff)); buff = []
        current_id = line[1:].split()[0]
        identifiers.append(current_id)
      else:
        buff.append(line)
    if current_id is not None:
      sequences.append(''.join(buff))
  return pd.DataFrame({'sequence_id': identifiers, 'sequence': sequences})

def write_fasta(df: pd.DataFrame, out_fasta: str) -> None:
  with open(out_fasta, 'w', encoding='utf-8') as fh:
    for _, row in df.iterrows():
      fh.write(f">{row['sequence_id']}\n{row['sequence']}\n")

def _extract_phylum_from_lineage(lineage: str) -> str:
  if not isinstance(lineage, str) or not lineage.strip():
    return 'Unknown'
  tokens = [t.strip() for t in lineage.split(';') if t.strip()]
  if len(tokens) >= 2:
    ph = tokens[1]
    if ph and ph.lower() not in DOMAIN_TOKENS:
      return ph
    return 'Unknown'
  if len(tokens) == 1:
    return 'Unknown' if tokens[0].lower() in DOMAIN_TOKENS else tokens[0]
  return 'Unknown'

def read_table_s2(table_path: Optional[str]) -> Optional[pd.DataFrame]:
  if not table_path or not os.path.exists(table_path):
    return None
  ext = Path(table_path).suffix.lower()
  try:
    if ext == ".tsv":
      meta = pd.read_csv(table_path, sep='\t', dtype=str, engine='python')
    else:
      meta = pd.read_csv(table_path, sep=None, dtype=str, engine='python')
  except Exception:
    meta = pd.read_csv(table_path, dtype=str)

  lower_map = {c.lower(): c for c in meta.columns}
  acc_col = lower_map.get('protein accession') or lower_map.get('protein_accession')             or lower_map.get('sequence id') or lower_map.get('sequence_id')
  lin_col = lower_map.get('lineage')

  if acc_col is not None and lin_col is not None:
    tmp = meta[[acc_col, lin_col]].copy().rename(columns={acc_col: 'sequence_id', lin_col: 'Lineage'})
    tmp['phylum'] = tmp['Lineage'].apply(_extract_phylum_from_lineage).map(_normalize_actino)
    return tmp[['sequence_id','phylum']].dropna().drop_duplicates()

  for c in ['phylum','Phylum','tax_phylum','taxonomy_phylum']:
    if c in meta.columns:
      if 'sequence_id' in meta.columns:
        tmp = meta[['sequence_id', c]].copy()
      elif acc_col is not None:
        tmp = meta[[acc_col, c]].copy().rename(columns={acc_col:'sequence_id'})
      else:
        continue
      tmp = tmp.rename(columns={c:'phylum'})
      tmp['phylum'] = tmp['phylum'].map(_normalize_actino)
      return tmp.dropna().drop_duplicates()
  return None

def merge_phylum_info(seqs_df: pd.DataFrame, meta_df: Optional[pd.DataFrame]) -> pd.DataFrame:
  out = seqs_df.copy()
  if meta_df is None:
    out['phylum'] = 'Unknown'
    return out
  out = out.merge(meta_df[['sequence_id','phylum']].drop_duplicates(), on='sequence_id', how='left')
  out['phylum'] = out['phylum'].fillna('Unknown').map(_normalize_actino)
  return out

@dataclass
class SubsetParams:
  size: Optional[int] = None
  fraction: Optional[float] = None
  seed: int = 42

def apply_subset_random(seqs_df: pd.DataFrame, params: SubsetParams) -> Tuple[pd.DataFrame, str]:
  n = len(seqs_df)
  if params.size is None and params.fraction is None:
    return seqs_df.copy(), 'full'
  rng = random.Random(params.seed)
  if params.size is not None:
    take_n = min(n, int(params.size))
    label = f"size_{take_n}"
  else:
    take_n = max(1, min(n, int(round(n*float(params.fraction)))))
    label = f"fraction_{params.fraction:.3f}".replace('.', 'p')
  indices = list(range(n)); rng.shuffle(indices)
  chosen = sorted(indices[:take_n])
  out = seqs_df.iloc[chosen].copy()
  return out, label


In [4]:

# =============================
# External tools (MAFFT, MMseqs2) & identities
# =============================
def _which(cmd: str, override: Optional[str]=None) -> Optional[str]:
  if override and os.path.isfile(override):
    return override
  path = shutil.which(cmd)
  if path:
    return path
  conda = os.environ.get('CONDA_PREFIX')
  if conda:
    cand = os.path.join(conda, 'bin', cmd)
    if os.path.isfile(cand):
      return cand
  try:
    out = subprocess.check_output(['whereis', cmd], text=True).strip()
    parts = out.split()
    for p in parts[1:]:
      if os.path.isfile(p):
        return p
  except Exception:
    pass
  return None

def run_mafft_msa(seqs_df: pd.DataFrame, run_dir: str, mafft_bin: Optional[str]=None, mafft_opts: Optional[List[str]]=None) -> str:
  ensure_output_dir(run_dir)
  fasta_in  = os.path.join(run_dir, 'sequences_for_mafft.fasta')
  fasta_out = os.path.join(run_dir, 'sequences_mafft_aligned.fasta')
  write_fasta(seqs_df, fasta_in)
  if mafft_opts is None:
    mafft_opts = ['--auto']
  exe = _which('mafft', mafft_bin)
  if exe is None:
    raise RuntimeError('MAFFT binary not found. Set MAFFT_BIN or ensure it is in PATH/CONDA_PREFIX.')
  cmd = [exe] + list(mafft_opts) + [fasta_in]
  print('[MAFFT]', ' '.join(cmd))
  with open(fasta_out, 'w', encoding='utf-8') as fout:
    proc = subprocess.run(cmd, stdout=fout, stderr=subprocess.PIPE, text=True)
  if proc.returncode != 0:
    raise RuntimeError(f"MAFFT failed:\n{proc.stderr}")
  return fasta_out

def read_aligned_fasta_to_array(fa_aligned: str) -> Tuple[List[str], np.ndarray]:
  identifiers, aligned = [], []
  if HAS_BIO:
    for rec in SeqIO.parse(fa_aligned, 'fasta'):
      identifiers.append(str(rec.id)); aligned.append(str(rec.seq))
  else:
    with open(fa_aligned, 'r', encoding='utf-8') as fh:
      current_id, buff = None, []
      for line in fh:
        line = line.strip()
        if not line: continue
        if line.startswith('>'):
          if current_id is not None:
            aligned.append(''.join(buff)); buff = []
          current_id = line[1:].split()[0]; identifiers.append(current_id)
        else:
          buff.append(line)
      if current_id is not None:
        aligned.append(''.join(buff))
  arr = np.array([list(s) for s in aligned], dtype='<U1')
  return identifiers, arr

def identities_stream_to_csv(ids: List[str], arr: np.ndarray, out_csv: str, sample_sizes: List[int]) -> Dict[int, pd.DataFrame]:
  n = len(ids); total_pairs = n*(n-1)//2
  ensure_output_dir(os.path.dirname(out_csv))
  with open(out_csv, 'w', encoding='utf-8') as fh:
    fh.write('seq_i,seq_j,identity_percent\n')
    strides = {s: max(1, total_pairs // int(s)) for s in sample_sizes}
    buffers = {s: [] for s in sample_sizes}
    k = 0
    for i in range(n):
      Ai = arr[i]
      for j in range(i+1, n):
        Aj = arr[j]
        both = (Ai != '-') & (Aj != '-')
        denom = int(both.sum())
        identity = 0.0 if denom == 0 else 100.0 * int(((Ai == Aj) & both).sum()) / denom
        fh.write(f"{ids[i]},{ids[j]},{identity:.6f}\n")
        for s in sample_sizes:
          if (k % strides[s]) == 0:
            buffers[s].append((ids[i], ids[j], identity))
        k += 1
  out = {}
  for s, buf in buffers.items():
    df = pd.DataFrame(buf, columns=['seq_i','seq_j','identity_percent'])
    df.to_csv(os.path.join(os.path.dirname(out_csv), f"pairs_sample_{s}.csv"), index=False)
    out[s] = df
  return out

def mmseqs_prepare_db(fasta_path: str, run_dir: str, mmseqs_bin: Optional[str]=None) -> Tuple[str, str, str]:
  exe = _which('mmseqs', mmseqs_bin)
  if exe is None:
    raise RuntimeError('MMseqs2 binary not found. Set MMSEQS_BIN or ensure it is in PATH/CONDA_PREFIX.')
  db = os.path.join(run_dir, 'mmseqs_db')
  tmp = os.path.join(run_dir, 'mmseqs_tmp')
  ensure_output_dir(tmp)
  subprocess.run([exe,'createdb',fasta_path,db], check=True)
  subprocess.run([exe,'createindex',db,tmp], check=True)
  return exe, db, tmp

def mmseqs_cluster_for_cutoff(exe: str, db: str, tmp: str, run_dir: str, cutoff: float, extra_opts: Optional[List[str]]=None) -> str:
  outbase = os.path.join(run_dir, f"mmseqs_clu_{int(cutoff)}")
  out = outbase
  min_id = float(cutoff)/100.0
  cmd = [exe,'cluster', db, out, tmp, '--min-seq-id', str(min_id)]
  if extra_opts is None:
    extra_opts = ['--cov-mode','0']
  cmd += extra_opts
  print('[MMseqs2]', ' '.join(cmd))
  try:
    subprocess.run(cmd, check=True, capture_output=True, text=True)
    tsv = outbase + '.tsv'
    cmd_tsv = [exe,'createtsv', db, db, out, tsv]
    subprocess.run(cmd_tsv, check=True, capture_output=True, text=True)
    return tsv
  except subprocess.CalledProcessError as e:
    print('[MMseqs2][cluster][error]:', (e.stderr or e.stdout or str(e)))
    fasta_in = os.path.join(run_dir, 'sequences_for_mmseqs.fasta')
    easy_dir = os.path.join(run_dir, f"easy_{int(cutoff)}"); ensure_output_dir(easy_dir)
    easy_cmd = [exe,'easy-cluster', fasta_in, easy_dir, tmp, '--min-seq-id', str(min_id)] + extra_opts
    subprocess.run(easy_cmd, check=True, capture_output=True, text=True)
    for cand in [os.path.join(easy_dir, 'cluster.tsv'),
                 easy_dir + '_cluster.tsv',
                 os.path.join(easy_dir, 'easy_cluster.tsv')]:
      if os.path.exists(cand):
        return cand
    return outbase + '.tsv'

def parse_mmseqs_tsv_to_clusters(tsv_path: str, cutoff: float) -> pd.DataFrame:
  rows = []
  with open(tsv_path, 'r', encoding='utf-8') as fh:
    for line in fh:
      line = line.strip()
      if not line or line.startswith('#'): 
        continue
      parts = line.split('\t')
      if len(parts) < 2: 
        continue
      representative, member = parts[0], parts[1]
      rows.append({'sequence_id': member, 'cutoff': float(cutoff), 'cluster_id': representative})
  df = pd.DataFrame(rows)
  if df.empty: 
    return df
  df['cluster_size'] = df.groupby(['cutoff','cluster_id'])['sequence_id'].transform('size')
  return df

def generate_clusters_with_mmseqs(seqs_df: pd.DataFrame, run_dir: str, cutoffs: List[int], mmseqs_bin: Optional[str]=None, extra_opts: Optional[List[str]]=None) -> pd.DataFrame:
  fasta_in = os.path.join(run_dir, 'sequences_for_mmseqs.fasta')
  write_fasta(seqs_df, fasta_in)
  exe, db, tmp = mmseqs_prepare_db(fasta_in, run_dir, mmseqs_bin=mmseqs_bin)
  parts = []
  for c in cutoffs:
    tsv = mmseqs_cluster_for_cutoff(exe, db, tmp, run_dir, c, extra_opts=extra_opts)
    parts.append(parse_mmseqs_tsv_to_clusters(tsv, c))
  out = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=['sequence_id','cutoff','cluster_id','cluster_size'])
  out_csv = os.path.join(run_dir, 'clusters_multi_threshold.csv')
  out.to_csv(out_csv, index=False)
  print('[OK] clusters ->', out_csv)
  return out


In [5]:

# =============================
# Plot helpers & palettes
# =============================
def save_multiformat(base_path_no_ext: str) -> Dict[str, str]:
  outputs = {}
  base = os.path.splitext(base_path_no_ext)[0]
  path_svg = base + '.svg'
  path_png = base + '.png'
  path_tif = base + '.tiff'
  plt.tight_layout()
  plt.savefig(path_svg, bbox_inches='tight')
  plt.savefig(path_png, dpi=900, bbox_inches='tight')
  plt.savefig(path_tif, dpi=900, bbox_inches='tight')
  plt.close()
  outputs['svg'] = path_svg; outputs['png'] = path_png; outputs['tiff'] = path_tif
  return outputs

def distinct_palette(n: int):
  from matplotlib import cm
  base = []
  for name in ['tab20', 'tab20b', 'tab20c']:
    cmap = cm.get_cmap(name)
    base.extend([cmap(i) for i in range(cmap.N)])
  if n <= len(base):
    return base[:n]
  import colorsys
  extra = []
  for i in range(n - len(base)):
    h = (i * 0.61803398875) % 1.0
    s = 0.8
    v = 0.9
    r,g,b = colorsys.hsv_to_rgb(h, s, v)
    extra.append((r,g,b,1.0))
  return base + extra

def is_excluded_phylum(s: Optional[str]) -> bool:
  if not isinstance(s, str):
    return True
  sl = s.strip().lower()
  if sl == '':
    return True
  return ('unknown' in sl) or ('uncultured' in sl) or ('uncultivated' in sl) or ('uncultived' in sl)

def add_umap_annotation(ax, distance_method_text: str, sequence_subset_label: str, pair_sample_label: Optional[str] = None):
  lines = [f"Distance method: {distance_method_text}"]
  if sequence_subset_label:
    lines.append(f"Sequence subset: {sequence_subset_label}")
  if pair_sample_label:
    lines.append(f"Pair sample: {pair_sample_label}")
  text = "\n".join(lines)
  ax.text(0.99, 0.99, text, transform=ax.transAxes, ha='right', va='top',
          fontsize=9, bbox=dict(boxstyle='round,pad=0.35', facecolor='white', edgecolor='black', alpha=0.8))

def _slugify(s: str) -> str:
  s = s.lower()
  s = re.sub(r"[^a-z0-9]+", "-", s).strip("-")
  return s

def build_info_suffix(distance_method_text: Optional[str], subset_label: Optional[str], pair_sample_label: Optional[str]) -> str:
  tokens = []
  if distance_method_text:
    tokens.append(f"dist-{_slugify(distance_method_text)}")
  if subset_label:
    tokens.append(f"subset-{_slugify(subset_label)}")
  if pair_sample_label:
    tokens.append(f"pairs-{_slugify(pair_sample_label.replace(',', ''))}")
  return ("__" + "_".join(tokens)) if tokens else ""


In [6]:

# =============================
# Dendrogram coloring by Phylum
# =============================
from matplotlib.lines import Line2D

def draw_dendrogram_with_phylum_colors(dissimilarity_matrix: np.ndarray, sequence_ids: List[str], phylum_map: Dict[str,str], out_path_base: str):
  condensed = squareform(dissimilarity_matrix, checks=False)
  linkage_matrix = linkage(condensed, method='average')
  plt.figure(figsize=(12, 7.5))
  ax = plt.gca()

  leaf_phyla = [phylum_map.get(sid, 'Unknown') for sid in sequence_ids]
  use_phyla = [p for p in leaf_phyla if not is_excluded_phylum(p)]
  if len(use_phyla) == 0:
    dendrogram(linkage_matrix, labels=sequence_ids, leaf_rotation=90, leaf_font_size=6)
    save_multiformat(out_path_base); return

  uniq = sorted(set(use_phyla))
  colors = distinct_palette(len(uniq))
  palette = {p: colors[i] for i,p in enumerate(uniq)}
  default_gray = (0.6,0.6,0.6,1.0)

  dendrogram(linkage_matrix, labels=sequence_ids, leaf_rotation=90, leaf_font_size=6,
             color_threshold=0.0, link_color_func=lambda k: default_gray)

  for lbl in ax.get_xmajorticklabels():
    sid = lbl.get_text()
    ph = phylum_map.get(sid, 'Unknown')
    if not is_excluded_phylum(ph) and ph in palette:
      lbl.set_color(palette[ph])
    else:
      lbl.set_color(default_gray)

  handles = [Line2D([0],[0], marker='o', color='w', label=p, markerfacecolor=palette[p], markersize=6) for p in uniq]
  if handles:
    ax.legend(handles=handles, title='Phylum', bbox_to_anchor=(0.5, -0.12), loc='upper center', ncol=min(6, max(1,len(handles))))
  save_multiformat(out_path_base)


In [7]:

# =============================
# Identity-based UMAP (precomputed)
# =============================
def build_dissimilarity_matrix_from_pairs(chosen_ids: List[str], pairs_df_sample: pd.DataFrame) -> np.ndarray:
  n = len(chosen_ids)
  D = np.ones((n,n), dtype=float); np.fill_diagonal(D, 0.0)
  index = {sid:i for i, sid in enumerate(chosen_ids)}
  for _, row in pairs_df_sample.iterrows():
    a, b, idp = row['seq_i'], row['seq_j'], float(row['identity_percent'])
    if a in index and b in index:
      i, j = index[a], index[b]
      d = 1.0 - (idp/100.0)
      if d < D[i, j]:
        D[i, j] = D[j, i] = d
  return D

def plot_umap_from_pairs_sample(pairs_df_sample: pd.DataFrame, seqs_df: pd.DataFrame, out_dir: str,
                                pair_sample_label: str, sequence_subset_label: str,
                                annotate_on_canvas: bool, append_info_to_filename: bool) -> Dict[str,str]:
  ensure_output_dir(out_dir)
  if pairs_df_sample is None or pairs_df_sample.empty or not HAS_UMAP:
    warnings.warn('Empty pairs sample or UMAP unavailable — skipped identity-based UMAP.'); 
    return {}
  ids_all = pd.unique(pd.concat([pairs_df_sample['seq_i'], pairs_df_sample['seq_j']], ignore_index=True)).tolist()
  ids_all = [sid for sid in ids_all if sid in set(seqs_df['sequence_id'].tolist())]
  if len(ids_all) < 2:
    warnings.warn('Not enough unique IDs for UMAP.'); return {}
  phylum_map = seqs_df.set_index('sequence_id')['phylum'].to_dict()
  ids_filtered = [sid for sid in ids_all if not is_excluded_phylum(phylum_map.get(sid, 'Unknown'))]
  if len(ids_filtered) < 2:
    warnings.warn('No eligible phyla to plot after filtering.'); return {}
  D = build_dissimilarity_matrix_from_pairs(ids_filtered, pairs_df_sample)
  reducer = umap.UMAP(metric='precomputed', random_state=RANDOM_STATE, n_neighbors=20, min_dist=0.05)
  Z = reducer.fit_transform(D)
  phyla = sorted({phylum_map.get(sid) for sid in ids_filtered})
  colors = distinct_palette(len(phyla)) if len(phyla)>0 else [(0,0,0,1)]
  cmap = {p: colors[i] for i,p in enumerate(phyla)}
  point_colors = [cmap.get(phylum_map.get(sid), (0.5,0.5,0.5,1)) for sid in ids_filtered]
  plt.figure(figsize=(13.0, 13.0))
  ax = plt.gca()
  ax.scatter(Z[:,0], Z[:,1], c=point_colors, s=18, alpha=0.95, linewidths=0)
  ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')
  from matplotlib.lines import Line2D
  handles = [Line2D([0],[0], marker='o', color='w', label=p, markerfacecolor=cmap[p], markersize=6) for p in phyla]
  if handles:
    ax.legend(handles=handles, title='Phylum', bbox_to_anchor=(0.5, -0.12), loc='upper center', ncol=min(6, max(1,len(phyla))))

  distance_text = "Alignment dissimilarity (1 − identity)"
  if annotate_on_canvas:
    add_umap_annotation(ax, distance_method_text=distance_text, 
                        sequence_subset_label=sequence_subset_label.replace('_',' '),
                        pair_sample_label=pair_sample_label)

  base = os.path.join(out_dir, '04_umap_dissimilarity_by_phylum')
  if append_info_to_filename:
    suffix = build_info_suffix(distance_text, sequence_subset_label, pair_sample_label)
    base = base + suffix
  return save_multiformat(base)


In [8]:

# =============================
# Other plots (bar, lines, heatmap, cluster stats)
# =============================
def plot_bar_identity_distribution(pairs_df_sample: pd.DataFrame, out_dir: str) -> Dict[str,str]:
  ensure_output_dir(out_dir)
  if pairs_df_sample is None or pairs_df_sample.empty:
    warnings.warn('Empty pairs sample — skipped barplot.'); return {}
  identities = pairs_df_sample['identity_percent'].to_numpy(dtype=float)
  bins = np.arange(10,101,5)
  hist, edges = np.histogram(identities, bins=bins)
  centers = (edges[:-1] + edges[1:]) / 2.0
  plt.figure(figsize=(10.5,6.4))
  plt.bar(centers, hist, width=4.8, edgecolor='black')
  plt.xlabel('Identity (%)'); plt.ylabel('Pair count (sample)')
  out = os.path.join(out_dir, '01_bar_identity_10_100')
  return save_multiformat(out)

def _canonicalize_pairs(df_pairs: pd.DataFrame) -> pd.DataFrame:
  a = df_pairs[['seq_i','seq_j']].min(axis=1)
  b = df_pairs[['seq_i','seq_j']].max(axis=1)
  out = df_pairs.copy()
  out['a'] = a; out['b'] = b
  return out.drop(columns=['seq_i','seq_j']).drop_duplicates(['a','b'])

def intra_cluster_pairs_for_cutoff(pairs_df: pd.DataFrame, clusters_df: pd.DataFrame, cutoff: int) -> pd.DataFrame:
  if pairs_df.empty or clusters_df.empty:
    return pd.DataFrame(columns=['a','b','identity_percent','cluster_id'])
  P = _canonicalize_pairs(pairs_df)
  mem = clusters_df[clusters_df['cutoff']==cutoff][['sequence_id','cluster_id']].drop_duplicates()
  m = (
    P.merge(mem.rename(columns={'sequence_id':'a'}), on='a', how='inner')
     .merge(mem.rename(columns={'sequence_id':'b','cluster_id':'cluster_id_b'}), on='b', how='inner')
  )
  m = m[m['cluster_id']==m['cluster_id_b']]
  return m[['a','b','identity_percent','cluster_id']]

def _counts_intra_by_cutoff_deciles(pairs_df: pd.DataFrame, clusters_df: pd.DataFrame) -> pd.DataFrame:
  bins = np.arange(10, 101, 10)
  rows = []
  for c in [10,20,30,40,50,60,70,80,90]:
    intra = intra_cluster_pairs_for_cutoff(pairs_df, clusters_df, c)
    if intra.empty:
      continue
    vals = intra['identity_percent'].to_numpy(dtype=float)
    vals = vals[~np.isnan(vals)]
    if vals.size == 0:
      continue
    hist, edges = np.histogram(vals, bins=bins)
    for b_idx in range(len(hist)):
      rows.append({'cutoff': int(c), 'bin_left': float(edges[b_idx]), 'bin_right': float(edges[b_idx+1]), 'x_value': int(edges[b_idx]), 'count': int(hist[b_idx])})
  return pd.DataFrame(rows)

def plot_intra_identity_counts_by_cutoff_lines(pairs_df: pd.DataFrame, clusters_df: pd.DataFrame, out_dir: str) -> Dict[str,str]:
  ensure_output_dir(out_dir)
  df = _counts_intra_by_cutoff_deciles(pairs_df, clusters_df)
  if df.empty:
    warnings.warn('No intra-cluster counts to plot.'); return {}
  plt.figure(figsize=(11.5,7.8))
  x_ticks = sorted(df['x_value'].unique())
  for c in sorted(df['cutoff'].unique()):
    sub = df[df['cutoff']==c].sort_values('x_value')
    plt.plot(sub['x_value'], sub['count'], marker='o', label=f'{c}%')
  plt.xticks(x_ticks, [str(x) for x in x_ticks], rotation=0)
  plt.xlabel('Identity (%) — decile bins left edges (10, 20, 30, …)')
  plt.ylabel('Intra-cluster pair count')
  plt.legend(title='Cutoff', bbox_to_anchor=(0.5, -0.16), loc='upper center', ncol=7)
  out = os.path.join(out_dir, '06a_counts_intra_identity_bins_by_cutoff_lines_REALX')
  return save_multiformat(out)

def plot_intra_identity_counts_by_cutoff_heatmap(pairs_df: pd.DataFrame, clusters_df: pd.DataFrame, out_dir: str) -> Dict[str,str]:
  ensure_output_dir(out_dir)
  df = _counts_intra_by_cutoff_deciles(pairs_df, clusters_df)
  if df.empty:
    warnings.warn('No intra-cluster counts to plot.'); return {}
  piv = df.pivot_table(index='cutoff', columns='x_value', values='count', aggfunc='sum', fill_value=0).sort_index()
  xvals = list(piv.columns)
  plt.figure(figsize=(max(10.5, piv.shape[1]*0.46), 7.9))
  plt.imshow(piv.to_numpy(), aspect='auto')
  plt.colorbar(label='Count')
  plt.yticks(range(piv.shape[0]), piv.index)
  plt.xticks(range(piv.shape[1]), [str(x) for x in xvals], rotation=0)
  plt.xlabel('Identity (%) — decile bins (10, 20, 30, …)')
  plt.ylabel('Cutoff (%)')
  out = os.path.join(out_dir, '06b_counts_intra_identity_bins_by_cutoff_heatmap')
  return save_multiformat(out)


In [9]:

# =============================
# Part B: Word2Vec & reductions
# =============================
@dataclass
class W2VParams:
  k: int = 3
  vector_size: int = 200
  window: int = 5
  min_count: int = 1
  sg: int = 1
  epochs: int = 20
  workers: int = 4

def tokens_from_msa(ids: List[str], arr: np.ndarray, k: int, drop_gaps: bool=True) -> Dict[str, List[str]]:
  toks = {}
  for i, sid in enumerate(ids):
    s = ''.join(arr[i])
    if drop_gaps: 
      s = s.replace('-', '')
    if len(s) >= k:
      toks[sid] = [s[j:j+k] for j in range(0, len(s)-k+1)]
    else:
      toks[sid] = ['PAD']
  return toks

def train_w2v(tokens_by_seq: Dict[str, List[str]], p: W2VParams):
  if not HAS_GENSIM:
    warnings.warn('gensim not available — skipping Part B.'); 
    return None
  model = Word2Vec(
    sentences=list(tokens_by_seq.values()),
    vector_size=p.vector_size, window=p.window,
    min_count=p.min_count, sg=p.sg, workers=p.workers,
    epochs=p.epochs, seed=RANDOM_STATE
  )
  return model

def sentence_and_mean_embeddings(tokens_by_seq: Dict[str, List[str]], model, run_dir: str) -> pd.DataFrame:
  sent_dir = os.path.join(run_dir, 'w2v_sentences'); ensure_output_dir(sent_dir)
  rows = []
  for sid, toks in tokens_by_seq.items():
    vecs = [model.wv[t] for t in toks if t in model.wv]
    if len(vecs)==0:
      emb = np.zeros((1, model.vector_size), dtype=float)
    else:
      emb = np.stack(vecs, axis=0)
    np.save(os.path.join(sent_dir, f'{sid}.npy'), emb)
    mean_vec = emb.mean(axis=0)
    row = {'sequence_id': sid}
    for i, v in enumerate(mean_vec):
      row[f'f{i:03d}'] = float(v)
    rows.append(row)
  df = pd.DataFrame(rows)
  pd.DataFrame({'sequence_id': list(tokens_by_seq.keys()), 'num_kmers': [len(tokens_by_seq[s]) for s in tokens_by_seq.keys()]}).to_csv(os.path.join(run_dir,'w2v_sentence_index.csv'), index=False)
  return df

def cosine_vs_identity_plots(pairs_df_sample: pd.DataFrame, emb_df: pd.DataFrame, seqs_df: pd.DataFrame, out_dir: str) -> Tuple[Dict[str,str], Dict[str,str]]:
  ensure_output_dir(out_dir)
  feat_cols = [c for c in emb_df.columns if c.startswith('f')]
  if not feat_cols:
    warnings.warn('No W2V features — skipping cosine vs identity.'); return {}, {}
  M = emb_df.set_index('sequence_id')[feat_cols]
  ph = seqs_df.set_index('sequence_id')['phylum'].to_dict()
  rows = []
  for _, r in pairs_df_sample.iterrows():
    a, b = r['seq_i'], r['seq_j']
    if a in M.index and b in M.index:
      va = M.loc[a].to_numpy(float); vb = M.loc[b].to_numpy(float)
      na = np.linalg.norm(va); nb = np.linalg.norm(vb)
      cosv = np.nan if na == 0 or nb == 0 else float(np.dot(va, vb) / (na*nb))
      t = 'Intra-phylum' if ph.get(a,'Unknown') == ph.get(b,'Unknown') else 'Inter-phylum'
      rows.append({'identity_percent': float(r['identity_percent']), 'cosine': cosv, 'type': t})
  df = pd.DataFrame(rows).dropna()
  if df.empty:
    warnings.warn('No pairs with embeddings for cosine plot.'); return {}, {}
  outs = {}
  for t, fname in [('Intra-phylum','07_cosine_vs_identity_intra'),
                   ('Inter-phylum','08_cosine_vs_identity_inter')]:
    sub = df[df['type']==t]
    if sub.empty:
      outs[fname] = {}
      continue
    plt.figure(figsize=(10.2,7.4))
    hb = plt.hexbin(sub['identity_percent'], sub['cosine'], gridsize=50, mincnt=2)
    plt.xlabel('Real identity (%)'); plt.ylabel('Cosine similarity (W2V mean)')
    cb = plt.colorbar(hb); cb.set_label('Pair count')
    bins = np.arange(10, 101, 10)
    sub2 = sub.copy()
    sub2['bin'] = pd.cut(sub2['identity_percent'], bins=bins, include_lowest=True, right=False)
    grp = sub2.groupby('bin').agg(x=('identity_percent','mean'), y=('cosine','mean')).dropna()
    if not grp.empty:
      plt.plot(grp['x'], grp['y'], marker='o', linewidth=2)
    out = os.path.join(out_dir, fname)
    outs[fname] = save_multiformat(out)
  return outs.get('07_cosine_vs_identity_intra', {}), outs.get('08_cosine_vs_identity_inter', {})

def reduce_and_plot_w2v(df_mean: pd.DataFrame, method: str, out_dir: str, sequence_subset_label: str,
                        annotate_on_canvas: bool, append_info_to_filename: bool) -> Dict[str,str]:
  feat = [c for c in df_mean.columns if c.startswith('f')]
  X = df_mean[feat].to_numpy(float)
  meta = df_mean[['sequence_id','phylum']].reset_index(drop=True)
  filt_mask = ~meta['phylum'].apply(is_excluded_phylum)
  if filt_mask.sum() < 2:
    warnings.warn('No eligible phyla to plot after filtering.'); return {}
  Xf = X[filt_mask.to_numpy()]
  metaf = meta[filt_mask].reset_index(drop=True)

  if method=='tsne':
    reducer = TSNE(n_components=2, random_state=RANDOM_STATE, init='pca', learning_rate='auto', perplexity=30)
    Z = reducer.fit_transform(Xf)
    base = os.path.join(out_dir, '09_tsne_w2v_mean_by_phylum')
  else:
    if not HAS_UMAP:
      warnings.warn('UMAP not installed — skipping UMAP of W2V means.'); return {}
    reducer = umap.UMAP(n_components=2, random_state=RANDOM_STATE, n_neighbors=20, min_dist=0.05, metric='euclidean')
    Z = reducer.fit_transform(Xf)
    base = os.path.join(out_dir, '10_umap_w2v_mean_by_phylum')
  df_plot = pd.DataFrame({'x':Z[:,0],'y':Z[:,1],'phylum':metaf['phylum']})
  uniq = sorted(df_plot['phylum'].astype(str).unique())
  colors = distinct_palette(len(uniq)); color_map = {p: colors[i] for i,p in enumerate(uniq)}
  C = [color_map[p] for p in df_plot['phylum']]
  plt.figure(figsize=(12.5, 12.5))
  ax = plt.gca()
  ax.scatter(df_plot['x'], df_plot['y'], s=14, c=C, alpha=0.95, linewidths=0)
  ax.set_xlabel('UMAP-1' if method=='umap' else 't-SNE-1')
  ax.set_ylabel('UMAP-2' if method=='umap' else 't-SNE-2')
  from matplotlib.lines import Line2D
  handles = [Line2D([0],[0], marker='o', color='w', label=p, markerfacecolor=color_map[p], markersize=6) for p in uniq]
  if handles:
    ax.legend(handles=handles, title='Phylum', bbox_to_anchor=(0.5, -0.12), loc='upper center', ncol=min(6, max(1, len(uniq))))

  if method=='umap' and annotate_on_canvas:
    distance_text = "Euclidean on Word2Vec mean embeddings"
    add_umap_annotation(ax, distance_method_text=distance_text,
                        sequence_subset_label=sequence_subset_label.replace('_',' '),
                        pair_sample_label=None)

  if method=='umap' and append_info_to_filename:
    distance_text = "Euclidean on Word2Vec mean embeddings"
    suffix = build_info_suffix(distance_text, sequence_subset_label, None)
    base = base + suffix

  return save_multiformat(base)

def plot_tsne_umap_w2v_means(df_mean: pd.DataFrame, out_dir: str, sequence_subset_label: str,
                             annotate_on_canvas: bool, append_info_to_filename: bool) -> None:
  ensure_output_dir(out_dir)
  reduce_and_plot_w2v(df_mean, 'tsne', out_dir, sequence_subset_label, annotate_on_canvas=False, append_info_to_filename=False)
  reduce_and_plot_w2v(df_mean, 'umap', out_dir, sequence_subset_label, annotate_on_canvas=annotate_on_canvas, append_info_to_filename=append_info_to_filename)


In [10]:

# =============================
# Orchestration
# =============================
def run_pipeline(
  sequences_path: str,
  output_dir: str,
  table_s2_path: Optional[str] = None,
  subset: SubsetParams = SubsetParams(),
  mafft_bin: Optional[str] = None,
  mmseqs_bin: Optional[str] = None,
  mafft_opts: Optional[List[str]] = None,
  mmseqs_extra_opts: Optional[List[str]] = None,
  sample_sizes: List[int] = [600_000],
  run_part_b: bool = True,
  w2v_params: 'W2VParams' = None,
  annotate_on_canvas: bool = True,
  append_info_to_filename: bool = True,
) -> str:
  if w2v_params is None:
    w2v_params = W2VParams()
  ensure_output_dir(output_dir)

  print('[i] Reading sequences...')
  seqs_full = read_fasta_minimal(sequences_path)
  meta = read_table_s2(table_s2_path)
  seqs_full = merge_phylum_info(seqs_full, meta)

  seqs_df, subset_label = apply_subset_random(seqs_full, subset)
  run_dir = os.path.join(output_dir, f"subset_{subset_label}"); ensure_output_dir(run_dir)
  write_fasta(seqs_df, os.path.join(run_dir, 'input_subset.fasta'))
  seqs_df.to_csv(os.path.join(run_dir, 'subset_ids.csv'), index=False)
  seqs_df[['sequence_id','phylum']].to_csv(os.path.join(run_dir, 'subset_phylum_map.csv'), index=False)

  fa_aln = run_mafft_msa(seqs_df, run_dir, mafft_bin=mafft_bin, mafft_opts=mafft_opts or ['--auto'])
  ids, arr = read_aligned_fasta_to_array(fa_aln)
  pairs_csv = os.path.join(run_dir, 'pairs.csv')
  samples = identities_stream_to_csv(ids, arr, pairs_csv, sample_sizes=sample_sizes)

  # Full dissimilarity for dendrogram
  id_index = {sid:i for i, sid in enumerate(ids)}
  n = len(ids)
  D_full = np.ones((n,n), dtype=float); np.fill_diagonal(D_full, 0.0)
  full_pairs = pd.read_csv(pairs_csv)
  for _, r in full_pairs.iterrows():
    a, b, idp = r['seq_i'], r['seq_j'], float(r['identity_percent'])
    if a in id_index and b in id_index:
      i, j = id_index[a], id_index[b]
      d = 1.0 - (idp/100.0)
      if d < D_full[i,j]:
        D_full[i,j] = D_full[j,i] = d
  ph_map = seqs_df.set_index('sequence_id')['phylum'].to_dict()
  draw_dendrogram_with_phylum_colors(D_full, ids, ph_map, os.path.join(run_dir, '03_dendrogram_branches_by_phylum'))

  clusters_df = generate_clusters_with_mmseqs(seqs_df, run_dir, FIXED_CUTOFFS, mmseqs_bin=mmseqs_bin, extra_opts=mmseqs_extra_opts or ['--cov-mode','0'])

  for s, df_s in samples.items():
    subplots_dir = os.path.join(run_dir, f'plots_sample_{s}'); ensure_output_dir(subplots_dir)
    plot_bar_identity_distribution(df_s, subplots_dir)
    plot_umap_from_pairs_sample(df_s, seqs_df, subplots_dir, pair_sample_label=f"{s:,}", sequence_subset_label=subset_label,
                                annotate_on_canvas=annotate_on_canvas, append_info_to_filename=append_info_to_filename)
    plot_intra_identity_counts_by_cutoff_lines(df_s, clusters_df, subplots_dir)
    plot_intra_identity_counts_by_cutoff_heatmap(df_s, clusters_df, subplots_dir)

  def plot_cluster_size_vs_cutoff(clusters_df: pd.DataFrame, out_dir: str):
    if clusters_df.empty: return False
    agg = clusters_df.groupby(['cutoff','cluster_id'])['sequence_id'].count().reset_index(name='cluster_size')
    stat = agg.groupby('cutoff')['cluster_size'].agg(['mean','median', lambda x: np.percentile(x,95)]).reset_index()
    stat = stat.rename(columns={'<lambda_0>':'p95'})
    plt.figure(figsize=(9.6,6.0))
    plt.plot(stat['cutoff'], stat['mean'], marker='o', label='Mean')
    plt.plot(stat['cutoff'], stat['median'], marker='o', label='Median')
    plt.plot(stat['cutoff'], stat['p95'], marker='o', label='P95')
    plt.xlabel('Identity cutoff (%)'); plt.ylabel('Cluster size')
    plt.legend(); plt.grid(True, alpha=0.3)
    save_multiformat(os.path.join(out_dir, '04_cluster_size_vs_identity_cutoff'))

    largest = agg.groupby('cutoff')['cluster_size'].max().reset_index()
    plt.figure(figsize=(9.6,6.0))
    plt.plot(largest['cutoff'], largest['cluster_size'], marker='o')
    plt.xlabel('Identity cutoff (%)'); plt.ylabel('Largest cluster size')
    plt.grid(True, alpha=0.3)
    save_multiformat(os.path.join(out_dir, '05_largest_cluster_by_identity_cutoff'))
    return True

  plot_cluster_size_vs_cutoff(clusters_df, run_dir)

  if run_part_b and HAS_GENSIM:
    toks = tokens_from_msa(ids, arr, k=(w2v_params.k if hasattr(w2v_params, 'k') else 3), drop_gaps=True)
    model = train_w2v(toks, w2v_params)
    if model is not None:
      emb_df = sentence_and_mean_embeddings(toks, model, run_dir)
      emb_df = emb_df.merge(seqs_df[['sequence_id','phylum']], on='sequence_id', how='left').fillna({'phylum':'Unknown'})
      for s, df_s in samples.items():
        subplots_dir = os.path.join(run_dir, f'plots_sample_{s}'); ensure_output_dir(subplots_dir)
        cosine_vs_identity_plots(df_s, emb_df, seqs_df, subplots_dir)
      plot_tsne_umap_w2v_means(emb_df[['sequence_id','phylum'] + [c for c in emb_df.columns if c.startswith('f')]], run_dir, subset_label,
                               annotate_on_canvas=annotate_on_canvas, append_info_to_filename=append_info_to_filename)

  with open(os.path.join(run_dir, 'FIGURE_MANIFEST.txt'), 'w', encoding='utf-8') as fh:
    fh.write("01_bar_identity_10_100\tGlobal identity distribution (sample)\n")
    fh.write("03_dendrogram_branches_by_phylum\tDendrogram (alignment dissimilarity) with branch colors by phylum\n")
    fh.write("04_umap_dissimilarity_by_phylum\tUMAP of alignment dissimilarity (1−identity); annotation/filename suffix per options\n")
    fh.write("06a_counts_intra_identity_bins_by_cutoff_lines_REALX\tIntra-cluster pair counts (decile identity) by cutoff (lines)\n")
    fh.write("06b_counts_intra_identity_bins_by_cutoff_heatmap\tIntra-cluster pair counts — heatmap (identity deciles × cutoff)\n")
    fh.write("04_cluster_size_vs_identity_cutoff\tMean/Median/P95 of cluster size vs cutoff (subset-level)\n")
    fh.write("05_largest_cluster_by_identity_cutoff\tLargest cluster vs cutoff (subset-level)\n")
    fh.write("07_cosine_vs_identity_intra\tCosine (W2V mean) vs real identity — Intra-phylum\n")
    fh.write("08_cosine_vs_identity_inter\tCosine (W2V mean) vs real identity — Inter-phylum\n")
    fh.write("09_tsne_w2v_mean_by_phylum\tt-SNE of mean W2V vectors by phylum (subset-level)\n")
    fh.write("10_umap_w2v_mean_by_phylum\tUMAP of mean W2V vectors by phylum (subset-level); annotation/filename suffix per options\n")
  print('[OK] Done. Outputs in:', run_dir)
  return run_dir


In [ ]:

# =============================
# Run
# =============================
subset_params = SubsetParams(size=SUBSET_SIZE, fraction=SUBSET_FRACTION, seed=SUBSET_SEED)
w2v_params = W2VParams(k=W2V_K, vector_size=W2V_VECTOR_SIZE, window=W2V_WINDOW,
                       min_count=W2V_MIN_COUNT, sg=W2V_SG, epochs=W2V_EPOCHS, workers=W2V_WORKERS)

_ = run_pipeline(
  sequences_path=FASTA_PATH,
  output_dir=OUTPUT_DIR,
  table_s2_path=TABLE_S2_PATH,
  subset=subset_params,
  mafft_bin=MAFFT_BIN,
  mmseqs_bin=MMSEQS_BIN,
  mafft_opts=['--auto'],
  mmseqs_extra_opts=['--cov-mode','0'],
  sample_sizes=PAIR_SAMPLE_SIZES,
  run_part_b=RUN_PART_B,
  w2v_params=w2v_params,
  annotate_on_canvas=ANNOTATE_ON_CANVAS,
  append_info_to_filename=APPEND_INFO_TO_FILENAME
)


[i] Reading sequences...
[MAFFT] /usr/bin/mafft --auto results_faal_jupyter/subset_full/sequences_for_mafft.fasta
